# Day 41：LangGraph 状态机编排入门 (`v0.4.0-day41`)

本 Notebook 完整演示智枢平台 Day 41 的核心能力（零外部依赖、无阻塞 `input()`）：
1. 使用 `StateGraph`、`START`、`END`、`Annotated` 归约器构建自定义状态机
2. 使用 `add_conditional_edges` 与循环守卫（`recursion_limit`）实现有界循环
3. 使用 `to_mermaid()` 导出 `stateDiagram-v2` 拓扑结构
4. 运行内置 **ReAct 工具调用循环图**（`build_react_graph`）与 `stream()` 流式输出
5. 运行 **费用报销与高风险操作人工审批流**（`build_approval_graph`）

In [1]:
import operator
import sys
from pathlib import Path
from typing import Annotated, TypedDict

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "project_iterations" / "day_41").exists():
    REPO_ROOT = REPO_ROOT.parent.parent.parent
DAY_DIR = REPO_ROOT / "project_iterations" / "day_41"
if str(DAY_DIR) not in sys.path:
    sys.path.insert(0, str(DAY_DIR))

from zhishu_agent import (
    END,
    START,
    GraphRecursionError,
    StateGraph,
    build_approval_graph,
    build_react_graph,
)

print("已加载 Day 41 目录:", DAY_DIR)

已加载 Day 41 目录: /home/antigravity/ab6539763/-AI-Agent/project_iterations/day_41


In [2]:
# 1. 自定义 StateGraph：条件边 + 有界循环 + Annotated 归约器 + recursion_limit 守卫
class LoopState(TypedDict, total=False):
    count: int
    logs: Annotated[list[str], operator.add]

def step_node(state: dict) -> dict:
    nxt = state.get("count", 0) + 1
    return {"count": nxt, "logs": [f"step-{nxt}"]}

def loop_router(state: dict) -> str:
    return "continue" if state["count"] < 3 else "done"

g = StateGraph(LoopState)
g.add_node("worker", step_node)
g.add_edge(START, "worker")
g.add_conditional_edges("worker", loop_router, {"continue": "worker", "done": END})
loop_app = g.compile()

print("--- 自定义循环图 Mermaid ---")
print(loop_app.to_mermaid())

res = loop_app.invoke({"count": 0, "logs": []})
print("循环执行结果:", res)

try:
    loop_app.invoke({"count": 0, "logs": []}, config={"recursion_limit": 2})
except GraphRecursionError as exc:
    print("触发死循环保护:", exc)

--- 自定义循环图 Mermaid ---
stateDiagram-v2
    [*] --> worker
    worker --> worker : continue
    worker --> [*] : done
循环执行结果: {'count': 3, 'logs': ['step-1', 'step-2', 'step-3']}
触发死循环保护: 状态图执行步数达到上限 recursion_limit=2，疑似存在未收敛循环 (当前节点: worker)


In [3]:
# 2. ReAct 工具调用循环图演示（stream 流式输出）
react_app = build_react_graph(max_iterations=4)
print("--- ReAct 状态图 Mermaid ---")
print(react_app.to_mermaid())

print("\n--- 流式执行: 计算 25 * 4 ---")
for event in react_app.stream({"query": "请帮我计算 25 * 4", "messages": [], "iteration": 0}):
    print(event)

kb_result = react_app.invoke({"query": "一线城市差旅住宿标准是多少？", "messages": [], "iteration": 0})
print("\n知识库检索最终回答:", kb_result["final_answer"])

--- ReAct 状态图 Mermaid ---
stateDiagram-v2
    [*] --> reason
    tools --> reason
    reason --> tools : tools
    reason --> [*] : done

--- 流式执行: 计算 25 * 4 ---
{'reason': {'pending_tool': 'calculator', 'pending_tool_input': '25 * 4', 'messages': [{'role': 'assistant', 'content': 'Thought: 需要计算 25 * 4，调用 calculator'}]}}
{'tools': {'iteration': 1, 'pending_tool': None, 'pending_tool_input': None, 'messages': [{'role': 'tool', 'name': 'calculator', 'content': '100'}]}}
{'reason': {'pending_tool': None, 'pending_tool_input': None, 'final_answer': '已完成检索/计算，结果：100', 'messages': [{'role': 'assistant', 'content': '最终回答：100'}]}}

知识库检索最终回答: 已完成检索/计算，结果：星辰数科差旅规范：一线城市住宿上限 600 元/晚，二线城市 450 元/晚。


In [4]:
# 3. 费用报销与高风险操作人工审批流演示
approval_app = build_approval_graph(auto_approve_limit=1000.0)
print("--- 审批流 Mermaid ---")
print(approval_app.to_mermaid())

low_res = approval_app.invoke({
    "request_id": "REQ-001",
    "applicant": "陈默",
    "action_type": "expense",
    "amount": 480.0,
    "audit_log": [],
})
print("\n低额度报销:", low_res["approval_status"], "|", low_res["execution_result"])

high_approved = approval_app.invoke({
    "request_id": "REQ-002",
    "applicant": "陈默",
    "action_type": "expense",
    "amount": 3500.0,
    "human_decision": True,
    "audit_log": [],
})
print("高额度人工通过:", high_approved["approval_status"], "|", high_approved["execution_result"])

high_rejected = approval_app.invoke({
    "request_id": "REQ-003",
    "applicant": "陈默",
    "action_type": "delete_prod_db",
    "amount": 0.0,
    "audit_log": [],
})
print("高风险默认拦截:", high_rejected["approval_status"], "|", high_rejected["execution_result"])

--- 审批流 Mermaid ---
stateDiagram-v2
    [*] --> risk_assess
    auto_approve --> execute_action
    execute_action --> [*]
    reject_action --> [*]
    risk_assess --> auto_approve : low_risk
    risk_assess --> human_review : high_risk
    human_review --> execute_action : approved
    human_review --> reject_action : rejected

低额度报销: approved | 已执行请求 REQ-001 (expense)
高额度人工通过: approved | 已执行请求 REQ-002 (expense)
高风险默认拦截: rejected | 请求 REQ-003 已被驳回，终止执行
